In [8]:
import cobra
import pandas as pd

# Load the iML1515 model
model = cobra.io.load_model("iML1515")

# Dictionaries to store our results
single_enzyme_reactions = {}
promiscuous_enzyme_reactions = {}
enzyme_complex_reactions = {}
enzyme_promiscuity_count = {}  # To track which enzymes catalyze multiple reactions

# First pass: Identify which genes/enzymes catalyze multiple reactions
for reaction in model.reactions:
    if reaction.genes:
        for gene in reaction.genes:
            if gene.id in enzyme_promiscuity_count:
                enzyme_promiscuity_count[gene.id].append(reaction.id)
            else:
                enzyme_promiscuity_count[gene.id] = [reaction.id]

# Second pass: Categorize reactions based on their enzyme associations
for reaction in model.reactions:
    if not reaction.genes:
        continue  # Skip reactions without enzyme associations
        
    # Get the GPR rule as a string
    gpr_rule = reaction.gene_reaction_rule
    
    # Check if this reaction is catalyzed by a complex (contains 'and' in the rule)
    if ' and ' in gpr_rule.lower():
        enzyme_complex_reactions[reaction.id] = {
            'gpr_rule': gpr_rule,
            'genes': [gene.id for gene in reaction.genes]
        }
    else:
        # Check if any of the enzymes catalyzing this reaction is promiscuous
        promiscuous = False
        for gene in reaction.genes:
            if len(enzyme_promiscuity_count[gene.id]) > 1:
                promiscuous = True
                break
                
        if promiscuous:
            promiscuous_enzyme_reactions[reaction.id] = {
                'gpr_rule': gpr_rule,
                'genes': [gene.id for gene in reaction.genes],
                'other_reactions': [rxn for rxn in enzyme_promiscuity_count[gene.id] if rxn != reaction.id]
            }
        else:
            single_enzyme_reactions[reaction.id] = {
                'gpr_rule': gpr_rule,
                'genes': [gene.id for gene in reaction.genes]
            }



In [14]:
# Print summary
print(f"Total reactions with enzyme associations: {len(single_enzyme_reactions) + len(promiscuous_enzyme_reactions) + len(enzyme_complex_reactions)}")
print(f"Reactions catalyzed by single non-promiscuous enzymes: {len(single_enzyme_reactions)}")
print(f"Reactions catalyzed by promiscuous enzymes: {len(promiscuous_enzyme_reactions)}")
print(f"Reactions catalyzed by enzyme complexes: {len(enzyme_complex_reactions)}")

Total reactions with enzyme associations: 2266
Reactions catalyzed by single non-promiscuous enzymes: 497
Reactions catalyzed by promiscuous enzymes: 1456
Reactions catalyzed by enzyme complexes: 313


In [13]:
# If you're considering each gene to represent an enzyme
# Count unique genes in the model
unique_genes = set(gene for reaction in model.reactions for gene in reaction.genes)
enzyme_count = len(unique_genes)
print(f"Total enzymes in the model: {enzyme_count}")

Total enzymes in the model: 1516


In [12]:
genes = sum(len(reaction.genes) for reaction in model.reactions)
print(f"Total genes in the model: {genes}")

Total genes in the model: 4624


In [ ]:
import cobra
import pandas as pd
import requests
import re
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

def process_gem(model, uniprot_mapping_file=None):
    """
    Process a GEM to extract enzyme information.
    
    Parameters:
    -----------
    model : GEM
        Read GEM with the COBRA package
    uniprot_mapping_file : str, optional
        Path to a file mapping model gene IDs to UniProt IDs (if needed).
        
    Returns:
    --------
    dict
        A dictionary with enzyme IDs as keys and dictionaries with enzyme 
        information as values.
    """
    
    # Output dictionary
    enzyme_dict = {}
    
    # Load UniProt mapping if provided
    uniprot_mapping = {}
    if uniprot_mapping_file:
        uniprot_df = pd.read_csv(uniprot_mapping_file, sep='\t')
        uniprot_mapping = dict(zip(uniprot_df['model_gene_id'], uniprot_df['uniprot_id']))
    
    # Process each reaction in the model
    for reaction in model.reactions:
        # Skip reactions without genes
        if not reaction.genes:
            continue
        
        # Process the gene-protein-reaction association
        process_gpr(reaction, enzyme_dict, uniprot_mapping, model)
    
    # Fetch protein sequences for all enzymes
    #fetch_protein_sequences(enzyme_dict)
    
    # Classify enzyme types
    classify_enzyme_types(enzyme_dict)
    
    return enzyme_dict

def process_gpr(reaction, enzyme_dict, uniprot_mapping, model):
    """
    Process gene-protein-reaction association for a reaction.
    """
    # Extract GPR rule
    gpr = reaction.gene_reaction_rule
    
    # Simple case: single gene (homomeric enzyme)
    if len(reaction.genes) == 1:
        gene = list(reaction.genes)[0]
        gene_id = gene.id
        uniprot_id = uniprot_mapping.get(gene_id, gene_id)
        
        if uniprot_id not in enzyme_dict:
            enzyme_dict[uniprot_id] = {
                'protein_seq': None,
                'substrates': [],
                'reactions': [],
                'type': None  # Will be classified later
            }
        
        # Add reaction information
        enzyme_dict[uniprot_id]['reactions'].append(reaction.id)
        
        # Add substrate information
        substrates = [m.id for m in reaction.reactants]
        for substrate in substrates:
            if substrate not in enzyme_dict[uniprot_id]['substrates']:
                enzyme_dict[uniprot_id]['substrates'].append(substrate)
    
    # Complex case: multiple genes (potentially heteromeric complex)
    else:
        # Extract genes involved in the reaction
        genes = list(reaction.genes)
        gene_ids = [gene.id for gene in genes]
        
        # Check if it's an OR relationship (isozymes) or AND relationship (complex)
        if ' or ' in gpr.lower():
            # Isozymes case (promiscuous)
            for gene in genes:
                gene_id = gene.id
                uniprot_id = uniprot_mapping.get(gene_id, gene_id)
                
                if uniprot_id not in enzyme_dict:
                    enzyme_dict[uniprot_id] = {
                        'protein_seq': None,
                        'substrates': [],
                        'reactions': [],
                        'type': None  # Will be classified later
                    }
                
                # Add reaction information
                enzyme_dict[uniprot_id]['reactions'].append(reaction.id)
                
                # Add substrate information
                substrates = [m.id for m in reaction.reactants]
                for substrate in substrates:
                    if substrate not in enzyme_dict[uniprot_id]['substrates']:
                        enzyme_dict[uniprot_id]['substrates'].append(substrate)
        
        elif ' and ' in gpr.lower():
            # Complex case - create an entry for the complex
            complex_id = '_'.join(sorted(gene_ids))
            
            if complex_id not in enzyme_dict:
                enzyme_dict[complex_id] = {
                    'protein_seq': None,
                    'substrates': [],
                    'reactions': [],
                    'type': None,  # Will be classified later
                    'components': []  # Store component genes
                }
            
            # Add component genes
            for gene_id in gene_ids:
                uniprot_id = uniprot_mapping.get(gene_id, gene_id)
                if uniprot_id not in enzyme_dict[complex_id]['components']:
                    enzyme_dict[complex_id]['components'].append(uniprot_id)
            
            # Add reaction information
            enzyme_dict[complex_id]['reactions'].append(reaction.id)
            
            # Add substrate information
            substrates = [m.id for m in reaction.reactants]
            for substrate in substrates:
                if substrate not in enzyme_dict[complex_id]['substrates']:
                    enzyme_dict[complex_id]['substrates'].append(substrate)
        
        else:
            # Handle more complex GPR expressions
            # This would require a more sophisticated parser for complex GPR rules
            pass

def fetch_protein_sequences(enzyme_dict):
    """
    Fetch protein sequences from UniProt for all enzymes.
    """
    for enzyme_id, info in enzyme_dict.items():
        # Skip complex entries (they don't have sequences)
        if 'components' in info:
            continue
        
        # Try to fetch sequence from UniProt
        if re.match(r'^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$', enzyme_id):
            # This is a UniProt ID format
            try:
                url = f"https://www.uniprot.org/uniprot/{enzyme_id}.fasta"
                response = requests.get(url)
                if response.status_code == 200:
                    sequence = ''.join(response.text.split('\n')[1:])
                    enzyme_dict[enzyme_id]['protein_seq'] = sequence
            except:
                pass
        
        # For complex entries, try to fetch sequences for components
        if 'components' in info:
            component_seqs = []
            for component_id in info['components']:
                if re.match(r'^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$', component_id):
                    try:
                        url = f"https://www.uniprot.org/uniprot/{component_id}.fasta"
                        response = requests.get(url)
                        if response.status_code == 200:
                            sequence = ''.join(response.text.split('\n')[1:])
                            component_seqs.append(sequence)
                    except:
                        pass
            
            # Store component sequences
            info['component_seqs'] = component_seqs

def classify_enzyme_types(enzyme_dict):
    """
    Classify enzymes into types:
    1: homomeric
    2: promiscuous
    3: homomeric complex
    4: heteromeric complex
    """
    for enzyme_id, info in enzyme_dict.items():
        # Check if it's a complex
        if 'components' in info:
            # Check if all components are the same (homomeric complex)
            if len(set(info['components'])) == 1:
                info['type'] = '3'  # homomeric complex
            else:
                info['type'] = '4'  # heteromeric complex
        else:
            # Single enzyme
            # Check if it catalyzes multiple reactions (promiscuous)
            if len(info['reactions']) > 1:
                info['type'] = '2'  # promiscuous
            else:
                info['type'] = '1'  # homomeric
                


In [7]:
# Test
gem = cobra.io.load_model("iML1515")
enzyme_dict = process_gem(gem)

In [8]:
# Access enzyme information
for enzyme_id, info in enzyme_dict.items():
    print(f"Enzyme: {enzyme_id}")
    print(f"Type: {info['type']}")
    print(f"Reactions: {', '.join(info['reactions'])}")
    print(f"Substrates: {', '.join(info['substrates'])}")
    if info['type'] in ['3', '4']:  # Complex
        print(f"Components: {', '.join(info['components'])}")
    print()

Enzyme: b2066
Type: 2
Reactions: CYTDK2, URIK1, URIK2, CYTDK1
Substrates: cytd_c, gtp_c, atp_c, uri_c

Enzyme: b0238
Type: 2
Reactions: XPPT, HXPRT, GUAPRT
Substrates: prpp_c, xan_c, hxan_c, gua_c

Enzyme: b0125
Type: 2
Reactions: HXPRT, GUAPRT
Substrates: hxan_c, prpp_c, gua_c

Enzyme: b0474
Type: 2
Reactions: NDPK5, NDPK6, NDPK8, ADK4, NDPK7, ADK1, NDPK2, NDPK3, NDPK4, NDPK1, ADK3, DADK, ADNK1
Substrates: atp_c, dgdp_c, dudp_c, dadp_c, amp_c, itp_c, dcdp_c, udp_c, cdp_c, dtdp_c, gdp_c, gtp_c, damp_c, adn_c

Enzyme: b2518
Type: 2
Reactions: NDPK5, NDPK6, NDPK8, NDPK7, NDPK2, NDPK3, NDPK4, NDPK1
Substrates: atp_c, dgdp_c, dudp_c, dadp_c, dcdp_c, udp_c, cdp_c, dtdp_c, gdp_c

Enzyme: b3281
Type: 1
Reactions: SHK3Dr
Substrates: 3dhsk_c, h_c, nadph_c

Enzyme: b1692
Type: 2
Reactions: SHK3Dr, QUINDHyi, QUINDH
Substrates: 3dhsk_c, h_c, nadph_c, nadp_c, quin_c, nad_c

Enzyme: b1062
Type: 1
Reactions: DHORTS
Substrates: dhor__S_c, h2o_c

Enzyme: b1281
Type: 1
Reactions: OMPDC
Substrates: h_c, 

In [9]:
print(len(enzyme_dict.items()))

1393


In [ ]:
import cobra
import pandas as pd
import requests
import re
import os
import time
from bioservices import UniProt

def process_gem(model, uniprot_mapping_file=None, organism="E coli"):
    """
    Process a GEM in SBML format to extract enzyme information.
    
    Parameters:
    -----------
    sbml_file : str
        Path to the SBML file containing the GEM.
    uniprot_mapping_file : str, optional
        Path to a file mapping model gene IDs to UniProt IDs (if needed).
    organism : str, optional
        Organism name for UniProt queries (default: "E coli")
        
    Returns:
    --------
    dict
        A dictionary with enzyme IDs as keys and dictionaries with enzyme 
        information as values.
    """
    # Load the model
    
    # Initialize the output dictionary
    enzyme_dict = {}
    
    # Set taxon ID based on organism
    if organism == 'E coli':
        taxon_ID = '83333'
    elif organism == 'Yeast':
        taxon_ID = '4932'
    elif organism == 'L major':
        taxon_ID = '5664'
        # Modify gene reaction rules if needed (like in the provided code)
        for reaction in model.reactions:
            reaction.gene_reaction_rule = reaction.gene_reaction_rule.replace('_DOT_', '.')
    else:
        # Default to E. coli if not specified
        taxon_ID = '83333'
    
    # Load UniProt mapping if provided, or create one from model gene IDs
    uniprot_mapping = {}
    if uniprot_mapping_file and os.path.exists(uniprot_mapping_file):
        uniprot_df = pd.read_csv(uniprot_mapping_file, sep='\t')
        uniprot_mapping = dict(zip(uniprot_df['model_gene_id'], uniprot_df['uniprot_id']))
    else:
        # Extract all unique gene IDs from the model
        all_gene_ids = set()
        for reaction in model.reactions:
            for gene in reaction.genes:
                all_gene_ids.add(gene.id)
        
        print(f"Mapping {len(all_gene_ids)} gene IDs to UniProt IDs...")
        
        # Initialize UniProt service
        service = UniProt()
        
        # Map gene IDs to UniProt IDs and fetch sequences
        gene_info = {}
        for gene_id in all_gene_ids:
            try:
                # Try mapping based on gene name
                query = f"gene_exact:({gene_id}) AND taxonomy_id:({taxon_ID})"
                result = service.search(query, frmt="tab", columns="id,sequence")
                
                if result and "\n" in result:
                    lines = result.strip().split("\n")
                    if len(lines) > 1:  # Skip header
                        fields = lines[1].split("\t")
                        if len(fields) >= 2:
                            uniprot_id = fields[0]
                            sequence = fields[1] if len(fields) > 1 else None
                            uniprot_mapping[gene_id] = uniprot_id
                            gene_info[gene_id] = {
                                'uniprot_id': uniprot_id,
                                'sequence': sequence
                            }
                            continue
            except Exception as e:
                print(f"Error looking up {gene_id}: {e}")
            
            # If no mapping was found, use the original gene ID
            if gene_id not in uniprot_mapping:
                uniprot_mapping[gene_id] = gene_id
        
        # Save the mapping for future use if a filename was provided
        if uniprot_mapping_file:
            mapping_df = pd.DataFrame({
                'model_gene_id': list(uniprot_mapping.keys()),
                'uniprot_id': list(uniprot_mapping.values())
            })
            mapping_df.to_csv(uniprot_mapping_file, sep='\t', index=False)
            print(f"Created mapping file with {len(uniprot_mapping)} entries: {uniprot_mapping_file}")
    
    # Process each reaction in the model
    for reaction in model.reactions:
        # Skip reactions without genes
        if not reaction.genes:
            continue
        
        # Process the gene-protein-reaction association
        process_gpr(reaction, enzyme_dict, uniprot_mapping, model, taxon_ID)
    
    # Fetch protein sequences for all enzymes that don't have them yet
    fetch_protein_sequences(enzyme_dict, taxon_ID)
    
    # Classify enzyme types
    classify_enzyme_types(enzyme_dict)
    
    return enzyme_dict

def get_UniProt_sequence(gene_id, taxon_ID):
    """
    Fetch protein sequence from UniProt for a given gene ID.
    
    Parameters:
    -----------
    gene_id : str
        Gene ID to look up.
    taxon_ID : str
        Taxonomy ID for the organism.
        
    Returns:
    --------
    str or None
        Protein sequence if found, None otherwise.
    """
    try:
        service = UniProt()
        query = f"gene_exact:({gene_id}) AND taxonomy_id:({taxon_ID})"
        result = service.search(query, frmt="fasta")
        if result:
            sequence = result.split("\n", 1)[-1]
            sequence = sequence.replace("\n", "")
            return sequence.strip()
        return None
    except Exception as e:
        print(f"Error fetching sequence for {gene_id}: {e}")
        return None

def process_gpr(reaction, enzyme_dict, uniprot_mapping, model, taxon_ID):
    """
    Process gene-protein-reaction association for a reaction.
    """
    # Extract GPR rule
    gpr = reaction.gene_reaction_rule.lower()
    
    # Simple case: single gene (homomeric enzyme)
    if len(reaction.genes) == 1:
        gene = list(reaction.genes)[0]
        gene_id = gene.id
        uniprot_id = uniprot_mapping.get(gene_id, gene_id)
        
        if uniprot_id not in enzyme_dict:
            enzyme_dict[uniprot_id] = {
                'protein_seq': get_UniProt_sequence(gene_id, taxon_ID),
                'substrates': [],
                'reactions': [],
                'type': None  # Will be classified later
            }
        
        # Add reaction information
        if reaction.id not in enzyme_dict[uniprot_id]['reactions']:
            enzyme_dict[uniprot_id]['reactions'].append(reaction.id)
        
        # Add substrate information
        substrates = [m.id for m in reaction.reactants]
        for substrate in substrates:
            if substrate not in enzyme_dict[uniprot_id]['substrates']:
                enzyme_dict[uniprot_id]['substrates'].append(substrate)
    
    # Complex case: multiple genes (potentially heteromeric complex)
    else:
        # Extract genes involved in the reaction
        genes = list(reaction.genes)
        gene_ids = [gene.id for gene in genes]
        
        # Check if it's an OR relationship (isozymes) or AND relationship (complex)
        if ' or ' in gpr:
            # Isozymes case (promiscuous)
            for gene in genes:
                gene_id = gene.id
                uniprot_id = uniprot_mapping.get(gene_id, gene_id)
                
                if uniprot_id not in enzyme_dict:
                    enzyme_dict[uniprot_id] = {
                        'protein_seq': get_UniProt_sequence(gene_id, taxon_ID),
                        'substrates': [],
                        'reactions': [],
                        'type': None  # Will be classified later
                    }
                
                # Add reaction information
                if reaction.id not in enzyme_dict[uniprot_id]['reactions']:
                    enzyme_dict[uniprot_id]['reactions'].append(reaction.id)
                
                # Add substrate information
                substrates = [m.id for m in reaction.reactants]
                for substrate in substrates:
                    if substrate not in enzyme_dict[uniprot_id]['substrates']:
                        enzyme_dict[uniprot_id]['substrates'].append(substrate)
        
        elif ' and ' in gpr:
            # Complex case - create an entry for the complex
            complex_id = '_'.join(sorted(gene_ids))
            
            if complex_id not in enzyme_dict:
                enzyme_dict[complex_id] = {
                    'protein_seq': None,
                    'substrates': [],
                    'reactions': [],
                    'type': None,  # Will be classified later
                    'components': []  # Store component genes
                }
            
            # Add component genes
            for gene_id in gene_ids:
                uniprot_id = uniprot_mapping.get(gene_id, gene_id)
                if uniprot_id not in enzyme_dict[complex_id]['components']:
                    enzyme_dict[complex_id]['components'].append(uniprot_id)
            
            # Add reaction information
            if reaction.id not in enzyme_dict[complex_id]['reactions']:
                enzyme_dict[complex_id]['reactions'].append(reaction.id)
            
            # Add substrate information
            substrates = [m.id for m in reaction.reactants]
            for substrate in substrates:
                if substrate not in enzyme_dict[complex_id]['substrates']:
                    enzyme_dict[complex_id]['substrates'].append(substrate)
        
        else:
            # Handle more complex GPR expressions
            # This would require a more sophisticated parser for complex GPR rules
            pass

def fetch_protein_sequences(enzyme_dict, taxon_ID):
    """
    Fetch protein sequences from UniProt for all enzymes that don't have them yet.
    """
    service = UniProt()
    
    for enzyme_id, info in enzyme_dict.items():
        # Skip complex entries (they don't have sequences)
        if 'components' in info:
            continue
        
        # If sequence is already fetched, skip
        if info['protein_seq']:
            continue
        
        # Try to fetch sequence from UniProt
        if re.match(r'^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$', enzyme_id):
            # This is a UniProt ID format
            try:
                query = f"id:{enzyme_id}"
                result = service.search(query, frmt="fasta")
                if result:
                    sequence = result.split("\n", 1)[-1]
                    sequence = sequence.replace("\n", "")
                    info['protein_seq'] = sequence.strip()
            except Exception as e:
                print(f"Error fetching sequence for {enzyme_id}: {e}")
        
        # For complex entries, try to fetch sequences for components
        if 'components' in info:
            component_seqs = []
            for component_id in info['components']:
                # Try to get sequence from UniProt
                try:
                    if re.match(r'^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$', component_id):
                        query = f"id:{component_id}"
                    else:
                        query = f"gene_exact:({component_id}) AND taxonomy_id:({taxon_ID})"
                    
                    result = service.search(query, frmt="fasta")
                    if result:
                        sequence = result.split("\n", 1)[-1]
                        sequence = sequence.replace("\n", "")
                        component_seqs.append(sequence.strip())
                except Exception as e:
                    print(f"Error fetching sequence for component {component_id}: {e}")
            
            # Store component sequences
            if component_seqs:
                info['component_seqs'] = component_seqs

def classify_enzyme_types(enzyme_dict):
    """
    Classify enzymes into types:
    1: homomeric
    2: promiscuous
    3: homomeric complex
    4: heteromeric complex
    """
    for enzyme_id, info in enzyme_dict.items():
        # Check if it's a complex
        if 'components' in info:
            # Check if all components are the same (homomeric complex)
            if len(set(info['components'])) == 1:
                info['type'] = '3'  # homomeric complex
            else:
                info['type'] = '4'  # heteromeric complex
        else:
            # Single enzyme
            # Check if it catalyzes multiple reactions (promiscuous)
            if len(info['reactions']) > 1:
                info['type'] = '2'  # promiscuous
            else:
                info['type'] = '1'  # homomeric

# Example usage
if __name__ == "__main__":
    # Example usage
    gem = cobra.io.load_model("iML1515")
    enzyme_info = process_gem(gem, organism="E coli")
    
    # Access enzyme information
    for enzyme_id, info in enzyme_info.items():
        print(f"Enzyme: {enzyme_id}")
        print(f"Type: {info['type']}")
        print(f"Reactions: {', '.join(info['reactions'])}")
        print(f"Substrates: {', '.join(info['substrates'])}")
        if info['type'] in ['3', '4']:  # Complex
            print(f"Components: {', '.join(info['components'])}")
        print()

Mapping 1516 gene IDs to UniProt IDs...
Error looking up b4290: Incorrect value provided (tab)    Correct values are ['xlsx', 'fasta', 'json', 'gff', 'tsv']
Error looking up b1656: Incorrect value provided (tab)    Correct values are ['xlsx', 'fasta', 'json', 'gff', 'tsv']
Error looking up b1002: Incorrect value provided (tab)    Correct values are ['xlsx', 'fasta', 'json', 'gff', 'tsv']
Error looking up b3632: Incorrect value provided (tab)    Correct values are ['xlsx', 'fasta', 'json', 'gff', 'tsv']
Error looking up b1453: Incorrect value provided (tab)    Correct values are ['xlsx', 'fasta', 'json', 'gff', 'tsv']
Error looking up b2178: Incorrect value provided (tab)    Correct values are ['xlsx', 'fasta', 'json', 'gff', 'tsv']
Error looking up b0115: Incorrect value provided (tab)    Correct values are ['xlsx', 'fasta', 'json', 'gff', 'tsv']
Error looking up b2258: Incorrect value provided (tab)    Correct values are ['xlsx', 'fasta', 'json', 'gff', 'tsv']
Error looking up b3374: 

KeyboardInterrupt: 